# 70 · The package pseudospace for both datasets

**What this does.** The user froze Caleb's package `pseudospace_reconstruction` 0.1.0 (commit 59da04b) as the
method that builds the pseudospace. This notebook runs its stages 01–04 on our inputs for both datasets, exports
the PT units for the downstream loader, and describes how the new coordinate and labels agree with the old ones.
- The stages: bins → units, QC, two-pass Harmony with automatic labels, and the elastic principal curve.
- **AKI:** Ctrl1A2 and Ctrl1A4 (reference Control) against IR2A2 and IR2A4 (case IR), on the local v4
  segmentations. It uses Caleb's shipped AKI configuration with one addition: the package's DPT baseline.
- **Human vs mouse (hvm):** the same two control mice (reference Mouse) against HUK1_COR1 and HUK1_MED1 (case
  Human; both are healthy cortex).
  - Human genes are renamed to mouse symbols through the accepted 1:1 ortholog map between stage 01 and stage 02,
    and every gene not shared by both species is dropped.
  - Stage 02 runs through the package's Python route (`stage02_inputs` → `qc.run_qc` → `save_stage02`) on
    ortholog-space bin matrices, because its split test re-reads the bins.

**The frozen protocol.** `docs/results/pt-package-pseudospace.md`, section "Notebook 70 protocol: the package
pseudospace for both datasets", written before any private-data run.

**What is never done here.** No label overrides, no parameter tuning, and no selection by agreement with the old
coordinates. Every agreement number is descriptive.

**Outputs** (`results/package_pseudospace/`):
- `runs/{aki,hvm}/`: the package run folders, with stages 01–04 and their manifests.
- `{aki,hvm}/pt_units.h5ad`: the PT units, raw integer counts, the export contract.
- `{aki,hvm}/all_units_positions.csv`: every unit after QC.
- `{aki,hvm}/cluster_labels_for_review.csv`: the coarse-label checkpoint, which the coordinator confirms per
  cluster.
- `{aki,hvm}/tables/` and `figures/`.

In [ ]:
import argparse
import hashlib
import json
import os
import subprocess
import sys
import time
from pathlib import Path

NOTEBOOK_START = time.perf_counter()
start = Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()
project = next(p for p in (start, *start.parents) if (p / 'pseudospace').is_dir())
sys.path.insert(0, str(project))
parser = argparse.ArgumentParser(add_help=False)
parser.add_argument('--data-root', type=Path)
parser.add_argument('--results-root', type=Path)
parser.add_argument('--visium-root', type=Path)
args, _ = parser.parse_known_args()
data_root = Path(args.data_root or os.environ.get('PSEUDOSPACE_DATA_ROOT') or project / 'data').expanduser().resolve()
results_root = Path(args.results_root or os.environ.get('PSEUDOSPACE_RESULTS_ROOT') or project / 'results').expanduser().resolve()
visium_root = args.visium_root or os.environ.get('PSEUDOSPACE_VISIUM_ROOT')
if not visium_root:
    raise RuntimeError('The mouse Visium HD bins live outside the data root: set PSEUDOSPACE_VISIUM_ROOT or pass '
                       '--visium-root (see data/README.md).')
os.environ['PSEUDOSPACE_VISIUM_ROOT'] = str(Path(visium_root).expanduser().resolve())   # samples.csv expands it
# Optional: the scratch AKI trial run (same config without DPT), for the determinism check. Read only.
TRIAL_RUN = Path(os.environ['PSEUDOSPACE_PACKAGE_TRIAL_RUN']) if os.environ.get('PSEUDOSPACE_PACKAGE_TRIAL_RUN') else None

NOTEBOOK_LOGIC_VERSION = '70.package_pseudospace.1'   # Bump when a cached calculation changes.
EXTERNAL_VERSION, EXTERNAL_COMMIT, HARMONY_VERSION = '0.1.0', '59da04b', '2.0.5'
DATASETS = {'aki': 'aki', 'hvm': 'human_vs_mouse'}     # export name -> tracked config folder
CONFIGS = {ds: project / 'analysis' / 'package_configs' / folder for ds, folder in DATASETS.items()}
output = results_root / 'package_pseudospace'
RUNS = {ds: output / 'runs' / ds for ds in DATASETS}
SPECIES = {'Ctrl1A2': 'mouse', 'Ctrl1A4': 'mouse', 'IR2A2': 'mouse', 'IR2A4': 'mouse',
           'HUK1_COR1': 'human', 'HUK1_MED1': 'human'}
PT_SEGMENTS = ['PT-S1', 'PT-S2', 'PT-S3']
MIN_IOU = .5                 # old <-> new polygon pairs: IoU >= 0.5 and mutual best
CENTROID_TOL_PX = 1.         # old structure -> its polygon: centroid check, as io.segmentation.polygons_for_obs
NEAR_TIE_MARGIN = .1         # label flag: any margin below this (the package flags cross-family ties only)
SPECIES_EXCLUSIVE = .95      # label flag (hvm): a cluster this much one species
N_TOP_MARKERS = 10
ORTHOLOG_FILE = results_root / 'minimal_pt_scfates' / 'ortholog_map_used.csv'
OLD = {   # old structures (ids, labels, coordinate) and the segmentation their feature_index points into
    'hvm': {'name': 'SCF13', 'object': results_root / 'minimal_pt_scfates' / 'cross_species_pt_scfates.h5ad',
            'coordinate': results_root / 'pt_reconstruction_v2' / 'primary_pt_coordinate.csv',
            'segmentation': {'Ctrl1A2': 'Ctrl1A2_kept_tubules_labeled_fine.geojson',
                             'Ctrl1A4': 'Ctrl1A4_kept_tubules_labeled_fine.geojson',
                             'HUK1_COR1': 'HUK1_COR1_v2.geojson', 'HUK1_MED1': 'HUK1_MED1_v2.geojson'}},
    'aki': {'name': 'mouse-only PT DPT', 'object': results_root / 'mouse_only_v5' / 'all_mouse_tubules_scanpy_dpt.h5ad',
            'pass1': results_root / 'mouse_only_v5' / 'all_mouse_tubules_harmony_pass1.h5ad',
            'segmentation': {'Ctrl1A2': 'Ctrl_1A2_v4.geojson', 'Ctrl1A4': 'Ctrl_1A4_v4.geojson',
                             'IR2A2': 'IR_2A2_v4.geojson', 'IR2A4': 'IR_2A4_v4.geojson'}},
}
cache = output / 'stage_cache'
for ds in DATASETS:
    for folder in (output / ds / 'tables', output / ds / 'figures'):
        folder.mkdir(parents=True, exist_ok=True)
REQUIRED = [ORTHOLOG_FILE, OLD['hvm']['object'], OLD['hvm']['coordinate'], OLD['aki']['object'], OLD['aki']['pass1'],
            *[data_root / f for ds in OLD for f in OLD[ds]['segmentation'].values()],
            *[data_root / 'tubule_by_gene' / f'{s}_tubule_by_gene_caleb.h5ad' for s in OLD['hvm']['segmentation']],
            *[CONFIGS[ds] / f for ds in CONFIGS for f in ('samples.csv', 'dataset.yaml', 'marker_panels.yaml')]]
for path in REQUIRED:
    if not path.is_file():
        raise FileNotFoundError(f'Missing input: {path} (notebooks 13 and 36 and the mouse_only_v5 workflow)')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml
from importlib.metadata import distribution, version
from IPython.display import display

pd.set_option('display.width', 240, 'display.max_columns', 40, 'display.max_colwidth', 90, 'display.max_rows', 120)
plt.rcParams.update({'font.family': ['Liberation Sans', 'DejaVu Sans'], 'font.size': 6.5, 'axes.titlesize': 6.5,
                     'axes.titleweight': 'bold', 'axes.labelsize': 6.5, 'xtick.labelsize': 6, 'ytick.labelsize': 6,
                     'legend.fontsize': 6, 'legend.frameon': False, 'axes.spines.top': False,
                     'axes.spines.right': False, 'pdf.fonttype': 42, 'savefig.dpi': 300})
SEGMENT_COLORS = {'PT-S1': '#4C9BD3', 'PT-S2': '#5B8E55', 'PT-S3': '#D6B48A'}   # the package's display colours


def sha256(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(1 << 20), b''):
            digest.update(block)
    return digest.hexdigest()


def cli(*arguments, tail=25):
    """Run `pseudospace-recon` with this kernel's Python; print the end of its output; raise on failure."""
    command = [sys.executable, '-m', 'pseudospace_reconstruction.cli', *map(str, arguments)]
    result = subprocess.run(command, capture_output=True, text=True)
    lines = (result.stdout + result.stderr).strip().splitlines()
    print('\n'.join(lines[-tail:]))
    if result.returncode != 0:
        raise RuntimeError(f'pseudospace-recon {arguments[0]} failed with exit code {result.returncode}')
    return result.stdout

## Guards (not cached)

These checks run first:
- the package version and commit, and R harmony 2.0.5;
- `validate` on both tracked configurations;
- the properties the package reads from every segmentation;
- the run record.

**The run record.** Each run folder holds `notebook70_record.json` with the digests of the configuration files
and segmentations it was built from. A run folder whose record differs from the current inputs is refused rather
than silently reused. To rebuild, delete that run folder.

In [ ]:
direct_url = json.loads(distribution('pseudospace-reconstruction').read_text('direct_url.json') or '{}')
EXTERNAL = {'version': version('pseudospace-reconstruction'),
            'commit': direct_url.get('vcs_info', {}).get('commit_id', 'unknown'), 'url': direct_url.get('url', '')}
assert EXTERNAL['version'] == EXTERNAL_VERSION and EXTERNAL['commit'].startswith(EXTERNAL_COMMIT), EXTERNAL
harmony = subprocess.run([sys.executable, '-c', (
    'import os, sys; os.environ.setdefault("R_HOME", os.path.join(sys.prefix, "lib", "R")); '
    'import rpy2.robjects as ro; print(ro.r("as.character(packageVersion(\'harmony\'))")[0])')],
    capture_output=True, text=True).stdout.strip().splitlines()
assert harmony and harmony[-1] == HARMONY_VERSION, f'R harmony must be {HARMONY_VERSION}: {harmony}'

from pseudospace_reconstruction import config as pcfg, pipeline, qc as pqc
from pseudospace_reconstruction.io import segmentation as pseg

from pseudospace.package_pseudospace import geojson_schema

DATASET = {ds: pcfg.load_config(CONFIGS[ds], data_root) for ds in DATASETS}
PATHS = {ds: pcfg.RunPaths(RUNS[ds]) for ds in DATASETS}
for ds in DATASETS:
    print(f'--- validate {ds}')
    cli('validate', '--config', CONFIGS[ds], '--data-root', data_root, tail=4)

SCHEMA = pd.DataFrame({s.segmentation_path.name: geojson_schema(s.segmentation_path)
                       for ds in DATASETS for s in DATASET[ds].samples}).T
display(SCHEMA)
assert (SCHEMA.n_with_class == SCHEMA.n_features).all() and (SCHEMA.n_with_score == SCHEMA.n_features).all(), \
    'A segmentation lacks classification or Score: stop (protocol).'
# No file carries properties.id: the package names units unit_<feature_index> in every sample (as in the trial).

SEG_SHA = {s.segmentation_path.name: sha256(s.segmentation_path) for ds in DATASETS for s in DATASET[ds].samples}
CONFIG_SHA = {ds: {f: sha256(CONFIGS[ds] / f) for f in ('samples.csv', 'dataset.yaml', 'marker_panels.yaml')}
              for ds in DATASETS}
RECORD = {ds: {'notebook_logic_version': NOTEBOOK_LOGIC_VERSION, 'package': EXTERNAL, 'config_sha256': CONFIG_SHA[ds],
               'segmentation_sha256': {s.name: SEG_SHA[s.segmentation_path.name] for s in DATASET[ds].samples},
               'bins': {s.name: str(s.bins_dir.relative_to(data_root)) if s.bins_dir.is_relative_to(data_root)
                        else f'$PSEUDOSPACE_VISIUM_ROOT/{s.bins_dir.relative_to(os.environ["PSEUDOSPACE_VISIUM_ROOT"])}'
                        for s in DATASET[ds].samples}} for ds in DATASETS}
for ds in DATASETS:
    record_path = RUNS[ds] / 'notebook70_record.json'
    started = any(PATHS[ds].done(k) for k in pcfg.STAGES)
    if record_path.exists():
        if json.loads(record_path.read_text()) != RECORD[ds]:
            raise RuntimeError(f'{RUNS[ds]} was built from other inputs or code; delete it to rebuild.')
    elif started:
        raise RuntimeError(f'{RUNS[ds]} holds stages without a notebook-70 record; delete it to rebuild.')
    else:
        RUNS[ds].mkdir(parents=True, exist_ok=True)
        record_path.write_text(json.dumps(RECORD[ds], indent=1))
print('guards passed:', EXTERNAL)

## AKI: stages 01–04 through the command line

The package skips a stage that already has a manifest (`run` without `--force`). Stage 01 forks one worker per
sample and needs about 22 GB.

In [ ]:
cli('run', '01', '02', '03', '04', '--config', CONFIGS['aki'], '--data-root', data_root, '--out', RUNS['aki'])
cli('status', '--out', RUNS['aki'], tail=12)

## Human vs mouse: stage 01, the shared orthologs, stage 02 on them, then stages 03–04

**Stage 01** runs on the original bins, so each sample keeps its native gene names.

**The shared genes** (`package_pseudospace.shared_ortholog_genes`) are the accepted HCOP 1:1 pairs whose mouse
symbol is a stage-01 gene of both mice and whose human symbol is a stage-01 gene of both human slices.

**Stage 02** sees:
- the units restricted to those genes, under mouse symbols;
- bin matrices restricted the same way (`00_ortholog_space/bins/<sample>`, with the same barcodes and links to
  the original `spatial/`).

**Stages 03–04** then run through the command line, which resumes after stage 02.

In [ ]:
import anndata as ad

from pseudospace.package_pseudospace import h5_gene_names, shared_ortholog_genes, to_shared_genes, write_shared_bin_dir

cli('run', '01', '--config', CONFIGS['hvm'], '--data-root', data_root, '--out', RUNS['hvm'])
hvm_config, hvm_paths = DATASET['hvm'], PATHS['hvm']
ortholog_dir = RUNS['hvm'] / '00_ortholog_space'
(ortholog_dir / 'bins').mkdir(parents=True, exist_ok=True)


def stage01_genes(sample):
    units = ad.read_h5ad(hvm_paths.units(sample), backed='r')
    genes = pd.Index(units.var_names.astype(str))
    units.file.close()
    return genes


STAGE01_GENES = {s.name: stage01_genes(s.name) for s in hvm_config.samples}
orthologs = pd.read_csv(ORTHOLOG_FILE)                 # loaded as coordinate_inputs.pt_inputs loads it
SHARED = shared_ortholog_genes(orthologs, [STAGE01_GENES[s] for s in STAGE01_GENES if SPECIES[s] == 'mouse'],
                               [STAGE01_GENES[s] for s in STAGE01_GENES if SPECIES[s] == 'human'])
SHARED.to_csv(ortholog_dir / 'shared_orthologs.csv', index=False)
panels = yaml.safe_load((CONFIGS['hvm'] / 'marker_panels.yaml').read_text())
panel_genes = sorted({g for genes in panels['label_panels'].values() for g in genes}
                     | {g for genes in panels['qc_regions']['markers'].values() for g in genes}
                     | set(panels['axis_anchors']['early']) | set(panels['axis_anchors']['late'])
                     | set(panels['extra_marker_genes']) | set(panels['extra_protected_genes']))
mouse_lookup = {g.upper() for s in STAGE01_GENES if SPECIES[s] == 'mouse' for g in STAGE01_GENES[s]}
NOT_SHARED = pd.DataFrame({'gene': [g for g in panel_genes if g not in set(SHARED.mouse_symbol)]})
NOT_SHARED['in_mouse_data'] = NOT_SHARED.gene.str.upper().isin(mouse_lookup)
NOT_SHARED['panels'] = NOT_SHARED.gene.map(lambda g: ', '.join(k for k, v in panels['label_panels'].items() if g in v)
                                           or ('extra_marker_genes' if g in panels['extra_marker_genes'] else ''))
NOT_SHARED.to_csv(output / 'hvm' / 'tables' / 'panel_genes_without_shared_ortholog.csv', index=False)
print(f'{len(SHARED):,} shared one-to-one orthologs of {len(orthologs):,} pairs; genes per sample:',
      {s: len(g) for s, g in STAGE01_GENES.items()})
display(NOT_SHARED)

BIN_DIRS = {}
for sample in hvm_config.samples:
    source_names = SHARED['mouse_symbol' if SPECIES[sample.name] == 'mouse' else 'human_source']
    h5_names = h5_gene_names(sample.bins_dir / 'filtered_feature_bc_matrix.h5')
    assert h5_names.equals(STAGE01_GENES[sample.name]), f'{sample.name}: stage-01 genes differ from its bin matrix'
    target = ortholog_dir / 'bins' / sample.name
    done = target / 'notebook70_bins.json'
    stamp = {'source_sha256': sha256(sample.bins_dir / 'filtered_feature_bc_matrix.h5'),
             'genes_sha256': hashlib.sha256('\n'.join(SHARED.mouse_symbol).encode()).hexdigest()}
    if not (done.exists() and json.loads(done.read_text()) == stamp):
        write_shared_bin_dir(sample.bins_dir, target, source_names, SHARED.mouse_symbol)
        done.write_text(json.dumps(stamp, indent=1))
    assert h5_gene_names(target / 'filtered_feature_bc_matrix.h5').equals(pd.Index(SHARED.mouse_symbol))
    BIN_DIRS[sample.name] = target

if not hvm_paths.done('02'):
    t0 = time.time()
    stage02 = pipeline.stage02_inputs(hvm_config, hvm_paths)
    shared_units = {s: to_shared_genes(units, SHARED, SPECIES[s]) for s, units in stage02['units'].items()}
    arguments = pipeline.stage02_arguments(hvm_config)
    arguments['bin_dirs'] = BIN_DIRS          # the only change to the package's stage-02 arguments
    qc_result = pqc.run_qc(shared_units, bin_tables=stage02['bin_tables'], **arguments)
    pipeline.save_stage02(hvm_config, hvm_paths, qc_result, time.time() - t0)
    del stage02, shared_units, qc_result
qc_genes = ad.read_h5ad(hvm_paths.qc_units, backed='r')
assert pd.Index(qc_genes.var_names).equals(pd.Index(SHARED.mouse_symbol)), 'stage 02 is not on the shared genes'
qc_genes.file.close()
(ortholog_dir / 'record.json').write_text(json.dumps({
    'ortholog_map': str(ORTHOLOG_FILE.relative_to(results_root)), 'ortholog_map_sha256': sha256(ORTHOLOG_FILE),
    'n_pairs': int(len(orthologs)), 'n_shared': int(len(SHARED)),
    'genes_per_sample': {s: int(len(g)) for s, g in STAGE01_GENES.items()},
    'stage02_route': 'pipeline.stage02_inputs -> to_shared_genes -> qc.run_qc(bin_dirs=00_ortholog_space/bins) '
                     '-> pipeline.save_stage02', 'notebook_logic_version': NOTEBOOK_LOGIC_VERSION}, indent=1))
cli('run', '03', '04', '--config', CONFIGS['hvm'], '--data-root', data_root, '--out', RUNS['hvm'])
cli('status', '--out', RUNS['hvm'], tail=12)

## Stage counts

In [ ]:
MANIFEST = {ds: {k: pcfg.read_manifest(PATHS[ds], k) for k in ('01', '02', '03', '04')} for ds in DATASETS}
POS = {ds: pd.read_parquet(PATHS[ds].pseudospace) for ds in DATASETS}
QC_FLAGS = {ds: pd.read_parquet(PATHS[ds].qc_flags, columns=['sample', 'unit_id', 'qc_retained']) for ds in DATASETS}
COUNTS = {}
for ds in DATASETS:
    binning = pd.read_csv(PATHS[ds].stage('01') / 'binning_summary.csv').set_index('sample')
    pos = POS[ds]
    pt = pos[pos.coarse_class.astype(str).eq('PT')]
    table = pd.DataFrame({
        'polygons': binning.n_polygons, 'with bins': binning.n_polygons_with_bins,
        'after QC': QC_FLAGS[ds].groupby('sample', observed=True).qc_retained.sum(),
        'pass 2 (curve)': pos.groupby('sample', observed=True).size(),
        **{seg: pt[pt.segment_class.astype(str).eq(seg)].groupby('sample', observed=True).size() for seg in PT_SEGMENTS}})
    table = table.fillna(0).astype(int)
    table.loc['total'] = table.sum()
    COUNTS[ds] = table
    table.to_csv(output / ds / 'tables' / 'stage_counts.csv')
    m3 = MANIFEST[ds]['03']['summary']
    print(f'== {ds}: genes after QC {MANIFEST[ds]["02"]["summary"]["n_genes"]:,}, pass 2 {m3["n_genes_pass2"]:,}; '
          f'clusters pass 1 {m3["n_clusters_pass1"]}, pass 2 {m3["n_clusters_pass2"]}; '
          f'dropped before pass 2 {m3["n_dropped_by_class"]}; family agreement {m3["family_agreement"]:.3f}, '
          f'ARI {m3["adjusted_rand"]:.3f}; QC discards {MANIFEST[ds]["02"]["summary"]["discarded_by_primary_reason"]}')
    display(table)

## Coarse-label checkpoint: every cluster of both passes

AGENTS.md requires an explicit confirmation of each cluster before a coordinate is used. Labels are the package's
automatic labels (`label_source` pass1); no override is applied.

**Top 10 markers.** These are the package's own ranking, rebuilt with its functions: Wilcoxon `rank_genes_groups`
on that pass's clustering feature space.
- Pass 1: `expression_gene_filter` → `clustering_feature_space` on `pass1_labelled.h5ad`.
- Pass 2: `clustering_feature_space` on `pass2.h5ad`. Its stored `selected_for_clustering` column is pass 1's,
  inherited from the earlier object.
- Their first five must equal the package's `pass*_cluster_top_markers.csv`.

**Flags.**
- `weak`: best score < 0.5.
- `ambiguous`: the package's cross-family tie.
- `near-tie`: any margin < 0.1.
- `dropped`: Unassigned, Rejected or a dropped class.
- `one-species` (hvm): more than 95% of the cluster's units come from one species.

In [ ]:
import scanpy as sc

from pseudospace.stage_cache import cached_frame
from pseudospace_reconstruction.integrate.cluster import clustering_feature_space, expression_gene_filter


def rank_markers(ds, which):
    """Top N_TOP_MARKERS Wilcoxon markers per cluster, as the package ranks them (see the markdown above)."""
    arguments = pipeline.stage03_arguments(DATASET[ds])
    marker_panels, params = arguments['panels'], arguments['params']
    if which == 'pass1':
        adata = ad.read_h5ad(PATHS[ds].pass1)
        adata = expression_gene_filter(adata, whitelist=marker_panels.gene_filter_whitelist(),
                                       min_fraction=params.min_gene_unit_fraction,
                                       min_total_counts=params.min_gene_total_counts,
                                       target_sum=params.normalize_target_sum, counts_layer='counts',
                                       lognorm_layer='lognorm')
        adata = clustering_feature_space(adata, whitelist=marker_panels.marker_whitelist(), n_top_genes=params.n_hvgs)
        key = 'leiden_coarse'
    else:
        # pass2.h5ad's var `selected_for_clustering` is pass 1's (inherited); the package re-selects on pass 2
        adata = clustering_feature_space(ad.read_h5ad(PATHS[ds].pass2), whitelist=marker_panels.marker_whitelist(),
                                         n_top_genes=params.n_hvgs)
        key = 'leiden_pass2'
    for name in list(adata.obsp):
        del adata.obsp[name]
    adata.uns.pop('rank_genes_groups', None)
    sc.tl.rank_genes_groups(adata, key, method='wilcoxon', pts=True)
    top = sc.get.rank_genes_groups_df(adata, group=None)
    top = top.sort_values(['group', 'scores'], ascending=[True, False]).groupby('group', observed=True).head(N_TOP_MARKERS)
    return top.groupby('group', observed=True)['names'].apply(lambda s: '; '.join(s)).rename('top10').reset_index()


def label_review(ds):
    rows, obs = [], {}
    for which in ('pass1', 'pass2'):
        stored = ad.read_h5ad(getattr(PATHS[ds], which), backed='r')
        obs[which] = stored.obs[['sample', 'leiden_coarse' if which == 'pass1' else 'leiden_pass2']].copy()
        stored.file.close()
    for which, key in (('pass1', 'leiden_coarse'), ('pass2', 'leiden_pass2')):
        folder = PATHS[ds].stage('03')
        labels = pd.read_csv(folder / f'{which}_cluster_labels.csv', index_col='cluster')
        labels.index = labels.index.astype(str)
        scores = pd.read_csv(folder / f'{which}_cluster_panel_scores.csv', index_col=0)
        scores.index = scores.index.astype(str)
        fingerprint = MANIFEST[ds]['03']['summary'][f'fingerprint_{which}']['membership_sha1']
        markers = cached_frame(f'top_markers_{ds}_{which}', lambda: rank_markers(ds, which), root=cache,
                               params={'n_top': N_TOP_MARKERS, 'feature_space': 're-selected per pass',
                                       'logic': NOTEBOOK_LOGIC_VERSION},
                               inputs={'fingerprint': fingerprint, 'run': RECORD[ds]}, code=rank_markers)
        markers = markers.assign(group=markers.group.astype(str)).set_index('group').top10
        per_sample = pd.crosstab(obs[which][key].astype(str), obs[which]['sample'].astype(str))
        for cluster, row in labels.iterrows():
            ranked = scores.loc[cluster].sort_values(ascending=False)
            top10 = markers.get(cluster, '')
            counts = per_sample.loc[cluster] if cluster in per_sample.index else pd.Series(dtype=int)
            species = counts.groupby(counts.index.map(SPECIES)).sum()
            flags = [name for name, on in [
                ('weak', bool(row.weak)), ('ambiguous', bool(row.ambiguous)),
                ('near-tie', row.margin < NEAR_TIE_MARGIN),
                ('dropped', row.family in set(DROPPED)),
                ('one-species', ds == 'hvm' and species.max() / max(species.sum(), 1) > SPECIES_EXCLUSIVE)] if on]
            rows.append({'pass': which, 'cluster': cluster, 'label': row.label, 'family': row.family,
                         'n_units': int(row.n_tubules), **{f'n_{s}': int(counts.get(s, 0)) for s in per_sample.columns},
                         **({'human_share': round(float(species.get('human', 0) / max(species.sum(), 1)), 3)}
                            if ds == 'hvm' else {}),
                         'panel_1': f'{ranked.index[0]} {ranked.iloc[0]:.2f}',
                         'panel_2': f'{ranked.index[1]} {ranked.iloc[1]:.2f}',
                         'panel_3': f'{ranked.index[2]} {ranked.iloc[2]:.2f}',
                         'margin': round(float(row.margin), 3), 'runner_up': row.runner_up,
                         'flags': ', '.join(flags), 'top10_markers': top10,
                         'top5_matches_package': top10.split('; ')[:5] == str(row.top_markers).split('; ')})
    return pd.DataFrame(rows)


DROPPED = [*panels['drop_classes'], 'Unassigned', *panels['rejected_labels'].values()]
REVIEW = {}
for ds in DATASETS:
    REVIEW[ds] = label_review(ds)
    REVIEW[ds].to_csv(output / ds / 'cluster_labels_for_review.csv', index=False)
    print(f'== {ds}: top-5 markers reproduce the package in {REVIEW[ds].top5_matches_package.mean():.0%} of clusters')
    display(REVIEW[ds].drop(columns=['top5_matches_package']))

## Stop conditions, orientation and root

Notebook 70 stops here, with no export, if any of these holds:
- a PT segment is missing in a specimen;
- the PT segment medians are not S1 < S2 < S3, pooled or in any specimen;
- the curve does not run S1 → S3.

`dpt_pseudotime` is the package's DPT baseline, a sensitivity coordinate. It is flipped if it runs S3 → S1, and
the flip is recorded.

In [ ]:
STOP, ORIENT = [], {}
for ds in DATASETS:
    pos = POS[ds]
    pt = pos[pos.coarse_class.astype(str).eq('PT')]
    medians = pt.groupby([pt['sample'].astype(str), pt.segment_class.astype(str)]).pseudospace.median().unstack()
    medians.loc['pooled'] = pt.groupby(pt.segment_class.astype(str)).pseudospace.median()
    for name, row in medians.iterrows():
        if row.reindex(PT_SEGMENTS).isna().any():
            STOP.append(f'{ds}/{name}: a PT segment is missing')
        elif not (row['PT-S1'] < row['PT-S2'] < row['PT-S3']):
            STOP.append(f'{ds}/{name}: PT segment medians out of order {row[PT_SEGMENTS].round(4).to_dict()}')
    root = MANIFEST[ds]['04']['summary']
    root_row = pos.loc[root['root_unit']]
    dpt = pt.dpt_pseudotime if 'dpt_pseudotime' in pt else pd.Series(np.nan, index=pt.index)
    dpt_medians = dpt.groupby(pt.segment_class.astype(str)).median()
    ORIENT[ds] = {'root_unit': root['root_unit'], 'root_rule': root['root_rule'],
                  'root_sample': str(root_row['sample']), 'root_label': str(root_row.segment_class),
                  'pt_medians': medians.loc['pooled', PT_SEGMENTS].round(4).to_dict(),
                  'runs_s1_to_s3': bool(medians.loc['pooled', 'PT-S1'] < medians.loc['pooled', 'PT-S3']),
                  'dpt_available': bool(dpt.notna().any()),
                  'dpt_flipped': bool(dpt.notna().any() and dpt_medians.get('PT-S1', 0) > dpt_medians.get('PT-S3', 0)),
                  'dpt_pt_nan': int(dpt.isna().sum())}
    if not ORIENT[ds]['runs_s1_to_s3']:
        STOP.append(f'{ds}: the curve runs S3 -> S1')
    print(f'== {ds}:', ORIENT[ds])
    display(medians.round(4))
if STOP:
    raise RuntimeError('Stop conditions (protocol): ' + '; '.join(STOP))

## Old structure ids: polygon matching

**Each old structure gets its source polygon** through its feature_index, verified by centroid (within 1 px, as
the package's `polygons_for_obs`). The sources:
- hvm: the SCF13 structures come from `data/tubule_by_gene`. The mouse ones are on
  `*_kept_tubules_labeled_fine.geojson` and the human ones on `HUK1_*_v2.geojson`.
- aki: the mouse-only v5 structures are on the v4 files, the same files as the new run.

**Old and new polygons are paired** when IoU ≥ 0.5 and each is the other's best match (`match_polygons`), over
all stage-01 polygons of the new run. Where both come from the same file, this should reduce to feature_index
identity.

In [ ]:
from pseudospace.package_pseudospace import match_polygons


def old_structures(ds):
    """Old structure id, sample, feature_index and centroid of every old polygon of one dataset."""
    if ds == 'hvm':
        parts = []
        for sample in OLD['hvm']['segmentation']:
            source = ad.read_h5ad(data_root / 'tubule_by_gene' / f'{sample}_tubule_by_gene_caleb.h5ad', backed='r')
            obs = source.obs[['sample', 'feature_index', 'x_centroid', 'y_centroid']].copy()
            source.file.close()
            obs.index = sample + '_' + obs.index.astype(str)
            parts.append(obs)
        return pd.concat(parts)
    source = ad.read_h5ad(OLD['aki']['pass1'], backed='r')
    obs = source.obs[['sample', 'feature_index', 'x_centroid', 'y_centroid']].copy()
    source.file.close()
    return obs


def match_dataset(ds):
    old = old_structures(ds)
    rows = []
    for sample in DATASET[ds].samples:
        name = sample.name
        o = old[old['sample'].astype(str).eq(name)]
        _, old_geometries = pseg.load_geometries(data_root / OLD[ds]['segmentation'][name])
        old_polygons = pseg.polygons_for_obs(o.astype({'feature_index': int}), old_geometries, tol_px=CENTROID_TOL_PX,
                                             label=f'old {name}')
        units = ad.read_h5ad(PATHS[ds].units(name), backed='r')
        new_obs = units.obs[['feature_index', 'x_centroid', 'y_centroid']].copy()
        units.file.close()
        _, new_geometries = pseg.load_geometries(sample.segmentation_path)
        new_polygons = pseg.polygons_for_obs(new_obs, new_geometries, tol_px=CENTROID_TOL_PX, label=f'new {name}')
        pairs = match_polygons(old_polygons, new_polygons, min_iou=MIN_IOU)
        rows.append(pd.DataFrame({
            'sample': name, 'new_unit': name + '_' + new_obs.index.astype(str).to_numpy()[pairs.new.to_numpy()],
            'new_feature_index': new_obs.feature_index.to_numpy(int)[pairs.new.to_numpy()],
            'old_structure_id': o.index.astype(str).to_numpy()[pairs.old.to_numpy()],
            'old_feature_index': o.feature_index.to_numpy(int)[pairs.old.to_numpy()],
            'iou': pairs.iou.to_numpy(), 'centroid_distance': pairs.centroid_distance.to_numpy(),
            'n_old': len(o), 'n_new': len(new_obs)}))
    return pd.concat(rows, ignore_index=True)


MATCH = {}
for ds in DATASETS:
    MATCH[ds] = cached_frame(f'polygon_match_{ds}', lambda ds=ds: match_dataset(ds), root=cache,
                             params={'min_iou': MIN_IOU, 'tol': CENTROID_TOL_PX, 'logic': NOTEBOOK_LOGIC_VERSION},
                             code=match_dataset,
                             inputs={'run': RECORD[ds], 'old': {s: sha256(data_root / f)
                                                              for s, f in OLD[ds]['segmentation'].items()},
                                     'old_obs': sha256(OLD[ds]['pass1']) if ds == 'aki' else
                                     {s: sha256(data_root / 'tubule_by_gene' / f'{s}_tubule_by_gene_caleb.h5ad')
                                      for s in OLD['hvm']['segmentation']}})
    MATCH[ds].to_csv(output / ds / 'tables' / 'polygon_matches.csv', index=False)
    m = MATCH[ds]
    summary = m.groupby('sample').agg(n_old=('n_old', 'first'), n_new=('n_new', 'first'), matched=('iou', 'size'),
                                      median_iou=('iou', 'median'), iou_1=('iou', lambda v: float((v > .999).mean())))
    summary['same_feature_index'] = m.groupby('sample').apply(
        lambda g: float((g.new_feature_index == g.old_feature_index).mean()), include_groups=False)
    print(f'== {ds}: old <-> new polygons (IoU >= {MIN_IOU}, mutual best)')
    display(summary.round(4))

## Exports: `pt_units.h5ad` and `all_units_positions.csv`

In [ ]:
from scipy import sparse

from pseudospace.package_pseudospace import uns_safe

OBS_COLUMNS = ['feature_index', 'x_centroid', 'y_centroid', 'seg_class', 'seg_score', 'n_spots', 'polygon_area_px',
               'area_um2', 'equivalent_diameter_um', 'bin_coverage', 'total_counts', 'n_genes_by_counts',
               'pct_counts_mt', 'pct_counts_ribo', 'log_total_counts', 'counts_per_bin', 'primary_identity',
               'identity_flag', 'doublet_tiers', 'coarse_class', 'broad_tubule_marker_call', 'leiden_coarse',
               'leiden_pass2', 'segment_class_pass2', 'celltype_margin']
CURVE_COLUMNS = ['curve_coordinate', 'curve_residual', 'curve_end_extension', 'curve_on_node', 'marker_axis']
OLD_IDS = {'hvm': pd.Index(ad.read_h5ad(OLD['hvm']['object'], backed='r').obs_names.astype(str)),
           'aki': pd.Index(ad.read_h5ad(OLD['aki']['object'], backed='r').obs_names.astype(str))}
UNITS, ALL_UNITS = {}, {}
for ds in DATASETS:
    pos = POS[ds]
    pass2_obs = ad.read_h5ad(PATHS[ds].pass2, backed='r')
    obs2, pass2_genes = pass2_obs.obs.copy(), pd.Index(pass2_obs.var_names)
    pass2_obs.file.close()
    pt_names = pos.index[pos.coarse_class.astype(str).eq('PT')]
    qc_units = ad.read_h5ad(PATHS[ds].qc_units)
    counts = sparse.csr_matrix(qc_units[pt_names].layers['counts'])
    assert np.allclose(counts.data, np.round(counts.data)) and (counts.data >= 0).all()
    match = MATCH[ds].set_index('new_unit')
    obs = pd.DataFrame(index=pd.Index(pt_names, name='unit'))
    obs['sample'] = pos.loc[pt_names, 'sample'].astype(str)
    obs['group'] = pos.loc[pt_names, 'condition'].astype(str)
    obs['segment_class'] = pos.loc[pt_names, 'segment_class'].astype(str)
    obs['pseudospace'] = pos.loc[pt_names, 'pseudospace'].astype(float)
    dpt = pos.loc[pt_names, 'dpt_pseudotime'].astype(float) if 'dpt_pseudotime' in pos else pd.Series(np.nan, index=pt_names)
    obs['dpt_pseudotime'] = 1 - dpt if ORIENT[ds]['dpt_flipped'] else dpt
    obs['polygon_id'] = obs2.loc[pt_names, 'original_obs_name'].astype(str)
    old_id = match.old_structure_id.reindex(pt_names)
    obs['old_structure_id'] = old_id.where(old_id.isin(OLD_IDS[ds])).astype(object)
    obs['old_match_iou'] = match.iou.reindex(pt_names).where(obs.old_structure_id.notna()).to_numpy(float)
    for column in OBS_COLUMNS:
        obs[column] = obs2.loc[pt_names, column].to_numpy()
    for column in CURVE_COLUMNS:
        obs[column] = pos.loc[pt_names, column].to_numpy()
    for column in obs.columns[obs.dtypes.eq(object)]:
        if column != 'old_structure_id':
            obs[column] = obs[column].astype(str)
    np.testing.assert_allclose(np.asarray(counts.sum(axis=1)).ravel(), obs.total_counts.to_numpy(float), rtol=1e-6)
    var = pd.DataFrame(index=pd.Index(qc_units.var_names.astype(str), name='gene'))
    for column in ('mt', 'ribo'):
        var[column] = qc_units.var[column].to_numpy(bool)
    var['in_pass2'] = var.index.isin(pass2_genes)
    if ds == 'hvm':
        var['human_symbol'] = SHARED.set_index('mouse_symbol').human_symbol.reindex(var.index).to_numpy()
        assert var.human_symbol.notna().all()
    del qc_units
    cfg_text = {f: (CONFIGS[ds] / f).read_text() for f in ('samples.csv', 'dataset.yaml', 'marker_panels.yaml')}
    samples_table = pd.read_csv(CONFIGS[ds] / 'samples.csv', comment='#')
    provenance = {
        'package': EXTERNAL, 'notebook': 'analysis/notebooks/70_pt_package_pseudospace.ipynb',
        'notebook_logic_version': NOTEBOOK_LOGIC_VERSION,
        'config_folder': str(CONFIGS[ds].relative_to(project)),
        'config': {'dataset_yaml': yaml.safe_load(cfg_text['dataset.yaml']),
                   'marker_panels_yaml': yaml.safe_load(cfg_text['marker_panels.yaml']),
                   'samples_csv': samples_table.to_dict(orient='list')},
        'config_sha256': CONFIG_SHA[ds],
        'segmentations': {s.name: {'file': s.segmentation_path.name, 'sha256': SEG_SHA[s.segmentation_path.name]}
                          for s in DATASET[ds].samples},
        'run_folder': str(RUNS[ds].relative_to(results_root)),
        'counts': 'raw integer counts of 02_qc/units_qc.h5ad layers[counts]: every gene kept through QC',
        'pt_units': "pass2.h5ad units with broad_tubule_marker_call == 'PT' (pass-1 labels, label_source pass1)",
        'pseudospace': 'stage 04 pseudospace (raw, [0, 1] over the nephron-wide curve), increasing S1 -> S3',
        'dpt_pseudotime': f"the package's stage-04 DPT baseline; flipped: {ORIENT[ds]['dpt_flipped']}",
        'root': ORIENT[ds],
        'old_structure_id': (f'{OLD[ds]["name"]} structure ids ({OLD[ds]["object"].relative_to(results_root)} '
                             f'obs index): polygon IoU >= {MIN_IOU}, mutual best, over every stage-01 polygon; old '
                             'polygons found through feature_index with a 1 px centroid check'),
        'fingerprints': {p: MANIFEST[ds]['03']['summary'][f'fingerprint_{p}'] for p in ('pass1', 'pass2')},
    }
    if ds == 'hvm':
        provenance['orthologs'] = {'map': str(ORTHOLOG_FILE.relative_to(results_root)), 'sha256': sha256(ORTHOLOG_FILE),
                                   'rule': 'HCOP one-to-one pairs measured in both species (stage-01 genes)',
                                   'n_shared': int(len(SHARED)),
                                   'stage02_route': 'stage02_inputs -> to_shared_genes -> run_qc(bin_dirs = '
                                                    'ortholog-space bins) -> save_stage02'}
    units = ad.AnnData(X=sparse.csr_matrix(counts, dtype=np.int32), obs=obs, var=var)
    units.uns['provenance'] = uns_safe(provenance)
    units.uns['provenance']['config_text'] = cfg_text
    units.write_h5ad(output / ds / 'pt_units.h5ad', compression='gzip')
    UNITS[ds] = units

    p1 = ad.read_h5ad(PATHS[ds].pass1, backed='r')
    all_units = p1.obs[['sample', 'segment_class', 'coarse_class']].copy()
    p1.file.close()
    all_units = all_units.rename(columns={'segment_class': 'label', 'coarse_class': 'family'})
    all_units['in_pass2'] = all_units.index.isin(pos.index)
    all_units['pseudospace'] = pos.pseudospace.reindex(all_units.index)
    all_units['dpt_pseudotime'] = pos.dpt_pseudotime.reindex(all_units.index) if 'dpt_pseudotime' in pos else np.nan
    all_units.index.name = 'unit'
    all_units.to_csv(output / ds / 'all_units_positions.csv')
    ALL_UNITS[ds] = all_units
    assert len(all_units) == MANIFEST[ds]['02']['summary']['n_retained']
    print(f'== {ds}: pt_units {units.shape}; all units {len(all_units):,} '
          f'({int(all_units.in_pass2.sum()):,} on the curve); old ids for {obs.old_structure_id.notna().mean():.1%} '
          f'of PT units')

## Descriptive checks

All of these are descriptive and never used to select anything:
- adjacent-segment AUCs within the PT;
- the stretch of the curve the PT occupies;
- Spearman correlation with the old coordinate on matched PT units, pooled and per specimen, also on notebook
  69's common-support subset;
- S1/S2/S3 agreement with the old labels.

The new curve is nephron-wide while the old coordinates were PT-only, so compare ranks within the PT, not
absolute values.

In [ ]:
from scipy.stats import spearmanr
from sklearn.metrics import cohen_kappa_score

from pseudospace.coordinate_inputs import common_support_mask, read_coordinate
from pseudospace.package_pseudospace import adjacent_aucs, pt_boundary


def old_coordinate(ds):
    """Old structure id -> (old coordinate, old segment label, in notebook 69's common support)."""
    if ds == 'hvm':
        source = ad.read_h5ad(OLD['hvm']['object'], backed='r')
        obs = source.obs[['sample', 'segment_class']].copy()
        source.file.close()
        obs = obs[source.obs.broad_tubule_marker_call.astype(str).eq('PT').to_numpy()]
        obs['old'] = read_coordinate(OLD['hvm']['coordinate']).reindex(obs.index.astype(str)).to_numpy()
        obs = obs[obs.old.notna()]
    else:
        source = ad.read_h5ad(OLD['aki']['object'], backed='r')
        obs = source.obs[['sample', 'segment_class', 'broad_tubule_marker_call', 'pt_subset_scanpy_dpt']].copy()
        source.file.close()
        obs = obs[obs.broad_tubule_marker_call.astype(str).eq('PT') & obs.segment_class.astype(str).isin(PT_SEGMENTS)]
        obs = obs.rename(columns={'pt_subset_scanpy_dpt': 'old'})
        obs = obs[obs.old.notna()]
    obs['support69'] = common_support_mask(obs.old.to_numpy(float), obs['sample'].astype(str).to_numpy())
    return obs[['sample', 'segment_class', 'old', 'support69']].astype({'sample': str, 'segment_class': str})


AGREE, ORDER, STRETCH, LABELS = {}, {}, {}, {}
for ds in DATASETS:
    pos, units = POS[ds], UNITS[ds]
    obs = units.obs
    # adjacent-segment AUCs within the PT
    rows = []
    for name, part in [('pooled', obs), *obs.groupby('sample', observed=True)]:
        for coordinate in ('pseudospace', 'dpt_pseudotime'):
            rows.append({'specimen': name, 'coordinate': coordinate,
                         **adjacent_aucs(part[coordinate].to_numpy(float), part.segment_class)})
    ORDER[ds] = pd.DataFrame(rows)
    # where the PT sits on the curve
    rows = []
    for name, part in [('pooled', pos), *pos.groupby('sample', observed=True)]:
        is_pt = part.coarse_class.astype(str).eq('PT').to_numpy()
        cut = pt_boundary(part.pseudospace.to_numpy(float), is_pt)
        ptv = part.pseudospace[is_pt]
        rows.append({'specimen': name, 'pt_units': int(is_pt.sum()), 'pt_q01': ptv.quantile(.01), 'pt_q50': ptv.median(),
                     'pt_q99': ptv.quantile(.99), **cut, 'pt_beyond_share': cut['pt_beyond'] / is_pt.sum()})
    STRETCH[ds] = pd.DataFrame(rows)
    # agreement with the old coordinate and labels on matched PT units
    old = old_coordinate(ds)
    print(f'{ds}: {len(old):,} old PT structures with a coordinate, {int(old.support69.sum()):,} in notebook 69\'s support')
    matched = obs[obs.old_structure_id.notna()].join(old, on='old_structure_id', rsuffix='_old', how='inner')
    rows = []
    for subset, frame in [('all matched', matched), ('notebook 69 support', matched[matched.support69.astype(bool)])]:
        for name, part in [('pooled', frame), *frame.groupby('sample', observed=True)]:
            rows.append({'subset': subset, 'specimen': name, 'n': len(part),
                         'spearman_pseudospace': spearmanr(part.pseudospace, part.old).statistic,
                         'spearman_dpt': spearmanr(part.dpt_pseudotime, part.old, nan_policy='omit').statistic})
    AGREE[ds] = pd.DataFrame(rows)
    LABELS[ds] = {
        'n_matched': len(matched),
        'identical': float((matched.segment_class == matched.segment_class_old).mean()),
        'kappa': float(cohen_kappa_score(matched.segment_class, matched.segment_class_old)),
        'per_specimen': matched.groupby('sample').apply(lambda g: float((g.segment_class == g.segment_class_old).mean()),
                                                        include_groups=False).round(3).to_dict(),
        'crosstab': pd.crosstab(matched.segment_class_old.rename('old'), matched.segment_class.rename('new'))}
    pd.crosstab([matched['sample'], matched.segment_class_old.rename('old')], matched.segment_class.rename('new')).to_csv(
        output / ds / 'tables' / 'label_crosstab_by_specimen.csv')
    # match rates in both directions
    pt_old = old.index
    reached = MATCH[ds].set_index('old_structure_id').new_unit.reindex(pt_old)
    LABELS[ds]['match_rates'] = {
        'new PT units with an old structure': float(obs.old_structure_id.notna().mean()),
        'old PT structures matched to a new polygon': float(reached.notna().mean()),
        'old PT structures matched to a new PT unit': float(reached.isin(obs.index).mean()),
        'old PT structures matched to a new unit on the curve': float(reached.isin(pos.index).mean())}
    fate = reached.map(lambda u: 'no new polygon' if pd.isna(u) else 'QC discarded' if u not in ALL_UNITS[ds].index
                       else ALL_UNITS[ds].label[u] + ('' if ALL_UNITS[ds].in_pass2[u] else ' (dropped in stage 03)'))
    LABELS[ds]['fate of old PT structures'] = fate.value_counts().to_dict()
    fate.groupby(old['sample'].to_numpy()).value_counts().unstack(fill_value=0).T.to_csv(
        output / ds / 'tables' / 'old_pt_structure_fate.csv')
    for name, table in [('order_aucs', ORDER[ds]), ('pt_stretch', STRETCH[ds]), ('coordinate_agreement', AGREE[ds]),
                        ('label_crosstab', LABELS[ds]['crosstab'])]:
        table.to_csv(output / ds / 'tables' / f'{name}.csv', index=name == 'label_crosstab')
    print(f'===== {ds} vs {OLD[ds]["name"]}')
    display(ORDER[ds].round(4), STRETCH[ds].round(4), AGREE[ds].round(4))
    print({k: v for k, v in LABELS[ds].items() if k != 'crosstab'})
    display(LABELS[ds]['crosstab'])

## AKI: determinism against the scratch trial

The trial ran the same configuration on the same machine, with DPT off. Stages 01–03 and the curve should be
identical; a difference is reported, not fixed. The check is skipped when `PSEUDOSPACE_PACKAGE_TRIAL_RUN` is
not set.

In [ ]:
TRIAL = {'available': TRIAL_RUN is not None and (TRIAL_RUN / '04_pseudospace' / 'manifest.json').exists()}
if TRIAL['available']:
    trial_paths = pcfg.RunPaths(TRIAL_RUN)
    for key in ('01', '02', '03', '04'):
        mine, theirs = MANIFEST['aki'][key]['summary'], pcfg.read_manifest(trial_paths, key)['summary']
        TRIAL[f'stage {key} summary identical'] = all(mine.get(k) == v for k, v in theirs.items())
    trial_pos = pd.read_parquet(trial_paths.pseudospace)
    same_units = trial_pos.index.equals(POS['aki'].index)
    TRIAL['same units on the curve'] = bool(same_units)
    if same_units:
        TRIAL['labels identical'] = bool((trial_pos.segment_class.astype(str) == POS['aki'].segment_class.astype(str)).all())
        TRIAL['max |delta pseudospace|'] = float(np.abs(trial_pos.pseudospace - POS['aki'].pseudospace).max())
print(TRIAL)

## Figures: the PT on each coordinate

In [ ]:
for ds in DATASETS:
    obs = UNITS[ds].obs
    old = old_coordinate(ds)
    matched = obs[obs.old_structure_id.notna()].join(old, on='old_structure_id', rsuffix='_old', how='inner')
    specimens = list(dict.fromkeys(obs['sample']))
    fig, axes = plt.subplots(1, 3, figsize=(180 / 25.4, 55 / 25.4))
    pos = POS[ds]
    for i, name in enumerate(specimens):
        part = pos[pos['sample'].astype(str).eq(name)]
        axes[0].hist(part.pseudospace, bins=np.linspace(0, 1, 101), histtype='step', lw=.7, label=name)
        for seg in PT_SEGMENTS:
            v = obs.pseudospace[(obs['sample'] == name) & (obs.segment_class == seg)]
            axes[1].boxplot(v, positions=[i * 4 + PT_SEGMENTS.index(seg)], widths=.7, showfliers=False,
                            patch_artist=True, boxprops={'facecolor': SEGMENT_COLORS[seg], 'lw': .5},
                            medianprops={'color': 'k', 'lw': .7}, whiskerprops={'lw': .5}, capprops={'lw': .5})
    cut = STRETCH[ds].set_index('specimen').loc['pooled', 'cut']
    axes[0].axvline(cut, color='k', lw=.5, ls='--')
    axes[0].set(xlabel='pseudospace (nephron-wide curve)', ylabel='units', title=f'{ds}: all units on the curve')
    axes[0].legend(loc='upper right')
    axes[1].set_xticks([i * 4 + 1 for i in range(len(specimens))], specimens, rotation=30, ha='right')
    axes[1].set(ylabel='pseudospace', title='PT-S1 / S2 / S3 per specimen')
    for seg in PT_SEGMENTS:
        m = matched[matched.segment_class_old == seg]
        axes[2].scatter(m.old, m.pseudospace, s=.3, color=SEGMENT_COLORS[seg], label=f'old {seg}', rasterized=True)
    rho = AGREE[ds].set_index(['subset', 'specimen']).loc[('all matched', 'pooled'), 'spearman_pseudospace']
    axes[2].set(xlabel=f'old coordinate ({OLD[ds]["name"]})', ylabel='pseudospace',
                title=f'matched PT units, Spearman {rho:.3f}')
    axes[2].legend(markerscale=10, loc='upper left')
    fig.tight_layout()
    fig.savefig(output / ds / 'figures' / 'pt_on_package_pseudospace.pdf')
    fig.savefig(output / ds / 'figures' / 'pt_on_package_pseudospace.png')
    plt.show()

## Summary

In [ ]:
SUMMARY = {ds: {'stage_counts': COUNTS[ds].to_dict(), 'orientation': ORIENT[ds],
                'labels_vs_old': {k: v for k, v in LABELS[ds].items() if k != 'crosstab'},
                'order_aucs': ORDER[ds].to_dict(orient='records'), 'pt_stretch': STRETCH[ds].to_dict(orient='records'),
                'coordinate_agreement': AGREE[ds].to_dict(orient='records'),
                'review_flags': REVIEW[ds]['flags'].value_counts().to_dict()} for ds in DATASETS}
SUMMARY['aki']['trial'] = TRIAL
SUMMARY['hvm']['orthologs'] = {'n_shared': int(len(SHARED)), 'panel_genes_without_shared_ortholog': NOT_SHARED.to_dict(orient='list')}
SUMMARY['notebook_logic_version'], SUMMARY['package'] = NOTEBOOK_LOGIC_VERSION, EXTERNAL
(output / 'summary.json').write_text(json.dumps(SUMMARY, indent=1, default=str))
print(f'notebook 70 finished in {(time.perf_counter() - NOTEBOOK_START) / 60:.1f} min')